# 07 — Candidate regulators (evidence-based ranking)

Replaces the template's `07_novel_drivers`, which ranks every gene in a disease-relevant regulon by how many MSigDB hallmark sets it belongs to, rewarding genes that sit in many hallmark sets in any cancer (GADD45B tops both GBM and KIRC). This notebook ranks **transcription factors**, the control points MINER actually proposes, using evidence specific to this cancer.

`{prefix}` = `CANCER.lower()`, `{val}` = validation cohort name in lower case. **The cancer and all paths are set in `config.py`; nothing in this notebook needs editing for a new cancer.**

## Evidence lines

| # | Evidence | Rule | Source |
|---|---|---|---|
| E1 | TF's regulons are enriched among disease-relevant regulons | hypergeometric test, FDR ≤ 0.05, at least 3 such regulons | 04 |
| E2 | Its survival-associated regulons point the same way | ≥ 3 survival-significant regulons, ≥ 80% all risk (HR > 1) or all protective (HR < 1) | 04 |
| E3 | Downstream of a driver mutation of this cancer | ≥ 1 filtered causal flow: driver mutation → TF → a disease-relevant regulon. Driver = IntOGen driver call for this cancer; tier **known** (also in the Cancer Gene Census) or **candidate** (not in it), from `driver_tiers.py` | 02 |
| E4 | TF's own expression predicts survival | univariate Cox on the TF's z-scored log2 TPM, FDR ≤ 0.05 | 01, 03 |
| E5 | Master regulator of at least one transcriptional state | present in `master_regulators_per_state.csv` | 06 |

**Ranking.** The TFs are ranked separately on each line (E1 enrichment p-value, E2 binomial p-value of risk vs protective regulons, E3 number of driver flows, E4 Cox p-value, E5 number of states) and the five rankings are combined with Robust Rank Aggregation (Kolde et al. 2012), which gives each TF a p-value (`rra_p`, `rra_fdr`) for sitting near the top more consistently than chance. The rules in the table are kept only as descriptive flags (`evidence_score`, 0–5); they do not set the rank.

**Tumour purity.** If `SI_geneset.gmt` (the ESTIMATE stromal and immune signatures, Yoshihara et al. 2013) is in `base_dir`, a stromal and an immune score are computed per tumour by ssGSEA (Barbie et al. 2009) and used as covariates in E4, so a TF is not credited for tracking the amount of non-tumour tissue. Without the file E4 is unadjusted and the notebook says so. Extra output: `stromal_immune_scores.csv`.

TFs that are not expressed (below `MIN_TPM` in more than `1 − MIN_FRAC` of tumours) are kept in the table but excluded from the ranking. Prior knowledge (Open Targets) is added at the end as an annotation; it does not affect the rank.

## Inputs

| File | From | Used for |
|---|---|---|
| `{prefix}_output/regulonDf.csv` | 02 | which TF controls which regulon |
| `{prefix}_output/regulon_cox_results.csv` | 04 | HR and p-value per regulon |
| `{prefix}_output/final_disease_relevant_regulons.csv` | 04 | disease-relevant set (no header) |
| `{prefix}_output/{val}_coexp_validated_regulons.csv` | 04 | regulons replicated in the independent cohort (no header) |
| `{prefix}_output/filteredCausalResults.csv` | 02 | mutation → TF → regulon flows |
| `{prefix}_output/master_regulators_per_state.csv`, `tf_network_degrees.csv` | 06 | master regulators, out/in ratio (optional) |
| `{prefix}_output/clinical_data.csv` | 03 | survival for the TF-level Cox test |
| `{prefix}_log2tpm.csv` | 01 | TF expression |
| `{prefix}_data/driver_genes.csv`, `open_targets_associations.csv` | `driver_tiers.py` (fetched once, then cached) | driver tiers for E3; Open Targets scores for the annotation |

## Outputs (in `{prefix}_output/`)

| File | Format |
|---|---|
| `candidate_regulators.csv` | one row per TF: regulon counts, enrichment p / FDR, direction, upstream driver mutations (all, and split into `upstream_known` / `upstream_candidate`), expression, own Cox HR / FDR, master-regulator states, E1–E5 flags, `evidence_score` (0–5), Open Targets score and status |
| `candidate_regulators_top20.png` | top 20 ranked TFs |

## What this does and does not show

These are **candidate regulators**, not drivers. "Driver" means a gene whose mutations are selected for in tumours, which only mutation-based methods can establish. Everything here is association in patient data. Functional evidence (for example DepMap CRISPR dependency in cell lines of this cancer) and tumour-vs-normal expression are the natural next columns to add.

> **GBM note.** TF expression and the TF-level survival test (E4) use the RNA-seq patients only (`gbm_log2tpm.csv`).

In [ ]:
# Necessary imports
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import miner.miner as miner
from scipy.stats import hypergeom, beta, binomtest
from lifelines import CoxPHFitter

warnings.filterwarnings("ignore")

# Repository root: holds config.py (paths and per-cancer settings) and driver_tiers.py
import sys
repo_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
import config

# CANCER CONFIGURATION: the cancer and the data root come from config.py
CANCER     = config.CANCER
base_dir   = config.BASE_DIR
prefix     = CANCER.lower()

# this cancer's own folder: holds <prefix>_data, <prefix>_notebooks and <prefix>_output
cancer_dir = os.path.join(base_dir, prefix)
data_dir   = os.path.join(cancer_dir, f"{prefix}_data")
output_dir = os.path.join(cancer_dir, f"{prefix}_output")

# Independent cohort used in 04 (same dictionary as 04 / 05)
VALIDATION_COHORTS = config.VALIDATION_COHORTS
HAS_VALIDATION = CANCER in VALIDATION_COHORTS
val = VALIDATION_COHORTS[CANCER][0].lower() if HAS_VALIDATION else "none"

# Driver gene tiers: published driver calls per cancer (module driver_tiers.py in the repository root, shared by 02, 06 and 07)
from driver_tiers import get_driver_tiers

# Driver genes of this cancer for evidence line E3: same table as 02 and 06
# both tiers need an IntOGen driver call for this cancer: "known" = also in the Cancer Gene Census, "candidate" = not in it
driver_table = get_driver_tiers(CANCER, data_dir)
tier_of = dict(zip(driver_table["Gene"], driver_table["Tier"]))

# A TF must reach MIN_TPM in at least MIN_FRAC of tumours to be ranked
MIN_TPM  = 1.0
MIN_FRAC = 0.20

assert os.path.isdir(output_dir), f"missing output folder: {output_dir}"
miner_dir  = os.path.dirname(miner.__file__)
idmap_path = os.path.join(miner_dir, "data", "identifier_mappings.txt")
print(f"Cancer: {CANCER} | outputs: {output_dir} | validation cohort: {val}")

In [ ]:
# Load the pipeline outputs

def bh_fdr(pvals):

    '''

    Benjamini-Hochberg FDR correction. NaN p-values are ignored and returned as NaN.

    Args: pvals (Series). Raw p-values, any index

    Returns: Series, same index. q-values

    '''

    p = pvals.dropna().sort_values()
    q = p * len(p) / np.arange(1, len(p) + 1)
    q = q[::-1].cummin()[::-1].clip(upper=1)
    return q.reindex(pvals.index)

def read_id_list(name):

    '''

    Reads a regulon list saved without a header (one regulon ID per line).

    Args: name (str). File name inside output_dir

    Returns: set of int regulon IDs

    '''

    return set(pd.read_csv(os.path.join(output_dir, name), header=None)[0].astype(int))

# Ensembl -> gene symbol
idmap = pd.read_csv(idmap_path, sep="\t", header=0)
names = idmap[idmap["Source"] == "Gene Name"]
ensembl_to_symbol = dict(zip(names["Preferred_Name"], names["Name"]))

# One row per (regulon, TF, gene); each regulon has exactly one TF
regulon_df = pd.read_csv(os.path.join(output_dir, "regulonDf.csv"), index_col=0)

# HR and p-value per regulon (index = regulon ID)
cox = pd.read_csv(os.path.join(output_dir, "regulon_cox_results.csv"), index_col=0)

# Disease-relevant regulons (final set of 04) and regulons replicated in the independent cohort
final_regulons     = read_id_list("final_disease_relevant_regulons.csv")
validated_regulons = read_id_list(f"{val}_coexp_validated_regulons.csv") if HAS_VALIDATION else set()

# Filtered causal flows: mutation -> TF (symbol) -> regulon ("R-123")
# Only the three columns used here are read (in heavily mutated cancers this file has millions of rows)
causal = pd.read_csv(os.path.join(output_dir, "filteredCausalResults.csv"),
                     usecols=["Mutation", "Regulator", "Regulon"], dtype="category")

# Keep flows from this cancer's driver genes only (all flows if there is no list), then go back to plain text columns
if tier_of:
    causal = causal[causal["Mutation"].isin(tier_of.keys())]
causal = causal.astype(str)
causal["Regulon_ID"] = causal["Regulon"].astype(str).str.replace("R-", "", regex=False).astype(int)

# Survival and expression for the TF-level tests
clinical = pd.read_csv(os.path.join(output_dir, "clinical_data.csv"), index_col=0)
log_tpm  = pd.read_csv(os.path.join(data_dir, f"{prefix}_log2tpm.csv"), index_col=0)

print(f"regulons: {regulon_df['Regulon_ID'].nunique()} | TFs: {regulon_df['Regulator'].nunique()} | "
      f"disease-relevant: {len(final_regulons)} | validated: {len(validated_regulons)} | causal flows: {len(causal)}")

In [ ]:
# Evidence E1 and E2: what each TF's regulons look like

# One row per regulon: its TF (as a symbol), survival statistics and set membership
reg = regulon_df[["Regulon_ID", "Regulator"]].drop_duplicates().set_index("Regulon_ID")
assert reg.index.is_unique, "a regulon is assigned to more than one TF"
reg["TF"] = reg["Regulator"].map(ensembl_to_symbol).fillna(reg["Regulator"])
reg = reg.join(cox[["HR", "p_value"]])
reg["survival_sig"] = reg["p_value"] <= 0.05
reg["risk"]         = reg["survival_sig"] & (reg["HR"] > 1)
reg["protective"]   = reg["survival_sig"] & (reg["HR"] < 1)
reg["validated"]    = reg.index.isin(validated_regulons)
reg["final"]        = reg.index.isin(final_regulons)

# Counts per TF
tf = reg.groupby("TF").agg(
    n_regulons=("TF", "size"),
    n_final=("final", "sum"),
    n_validated=("validated", "sum"),
    n_survival=("survival_sig", "sum"),
    n_risk=("risk", "sum"),
    n_protective=("protective", "sum"),
).astype(int)

# E1: does this TF hold more disease-relevant regulons than expected by chance?
# Hypergeometric test: drawing n_regulons from all regulons, probability of getting >= n_final disease-relevant ones
# Testing against the background rate means a TF is not rewarded just for having many regulons
n_all, n_all_final = len(reg), int(reg["final"].sum())
tf["frac_final"]   = tf["n_final"] / tf["n_regulons"]
tf["enrich_p"]     = hypergeom.sf(tf["n_final"] - 1, n_all, n_all_final, tf["n_regulons"])
tf["enrich_fdr"]   = bh_fdr(tf["enrich_p"])

# E2: do its survival-associated regulons agree in direction?
tf["direction_consistency"] = (tf[["n_risk", "n_protective"]].max(axis=1) / tf["n_survival"]).where(tf["n_survival"] > 0)
tf["direction"] = np.where(tf["n_risk"] > tf["n_protective"], "risk",
                  np.where(tf["n_protective"] > tf["n_risk"], "protective", "mixed"))

print(f"TFs: {len(tf)} | background rate of disease-relevant regulons: {n_all_final / n_all:.1%}")
print(tf.sort_values("enrich_p").head(10)[["n_regulons", "n_final", "frac_final", "enrich_fdr", "direction", "direction_consistency"]])

In [ ]:
# Evidence E3: is the TF downstream of a driver mutation of this cancer?

# Driver genes from driver_tiers.py; if there is no list for this cancer every mutation in the causal results counts
drivers = sorted(tier_of) if tier_of else sorted(set(causal["Mutation"]))
if not tier_of:
    print(f"No driver list for {CANCER}: using all {len(drivers)} mutated genes")

def joined(s):

    '''

    Joins the distinct values of a column into one comma-separated string.

    Args: s (Series). Gene symbols

    Returns: str. Sorted unique symbols separated by ", "

    '''

    return ", ".join(sorted(set(s)))

# Only flows that start at a driver AND end in a disease-relevant regulon count
causal["driver"] = causal["Mutation"].isin(drivers)
causal["Tier"]   = causal["Mutation"].map(tier_of).fillna("none")
causal["final"]  = causal["Regulon_ID"].isin(final_regulons)
strong_flows = causal[causal["driver"] & causal["final"]]

tf["n_causal_flows"]       = causal.groupby("Regulator").size().reindex(tf.index).fillna(0).astype(int)
tf["n_driver_flows_final"] = strong_flows.groupby("Regulator").size().reindex(tf.index).fillna(0).astype(int)
tf["upstream_drivers"]     = strong_flows.groupby("Regulator")["Mutation"].apply(joined).reindex(tf.index).fillna("")

# The same split by tier: flows from catalogued drivers and from statistically called but uncatalogued ones (the novel side)
for tier in ["known", "candidate"]:
    tier_flows = strong_flows[strong_flows["Tier"] == tier]
    tf[f"upstream_{tier}"] = tier_flows.groupby("Regulator")["Mutation"].apply(joined).reindex(tf.index).fillna("")

print("Flows per mutation (all | driver -> disease-relevant regulon):")
flow_table = pd.DataFrame({"all": causal["Mutation"].value_counts(), "strong": strong_flows["Mutation"].value_counts()}).fillna(0).astype(int)
flow_table["Tier"] = [tier_of.get(g, "none") for g in flow_table.index]
print(flow_table.sort_values("all", ascending=False).to_string())
print(f"TFs with at least one driver flow into a disease-relevant regulon: {int((tf['n_driver_flows_final'] > 0).sum())} "
      f"| via a candidate-tier driver: {int((tf['upstream_candidate'] != '').sum())}")


In [ ]:
# Stromal and immune content of each tumour (ESTIMATE approach, Yoshihara et al. 2013)
# Many survival-associated TFs are stromal or immune cell TFs: their signal can reflect how much non-tumour tissue is in the
# sample, not regulation inside tumour cells. The two scores are used as covariates in E4
# Needs the ESTIMATE signature file SI_geneset.gmt (141 stromal + 141 immune genes) in base_dir; skipped if it is missing

def ssgsea_scores(expr, gene_set, alpha=0.25):

    '''

    Single-sample gene set enrichment score (Barbie et al. 2009), the statistic ESTIMATE uses for its two signatures.

    Args: expr (df), genes x patients. Expression values (any monotonic scale)
          gene_set (list of str). Gene symbols of the signature
          alpha (float). Weight exponent on the gene ranks (default 0.25)

    Returns: Series, index = patients. Enrichment score, higher = signature genes rank higher in that sample

    '''

    in_set = expr.index.isin(gene_set)
    n_genes, scores = len(expr), {}
    for patient in expr.columns:
        order   = np.argsort(-expr[patient].values, kind="stable")
        hit     = in_set[order]
        weights = np.where(hit, np.arange(n_genes, 0, -1) ** alpha, 0.0)
        p_hit   = np.cumsum(weights) / weights.sum()
        p_miss  = np.cumsum(~hit) / (~hit).sum()
        scores[patient] = (p_hit - p_miss).sum()
    return pd.Series(scores)

gmt_path = os.path.join(base_dir, "SI_geneset.gmt")
if os.path.exists(gmt_path):
    signatures = {}
    with open(gmt_path) as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            signatures[parts[0]] = [g for g in parts[2:] if g]

    purity = pd.DataFrame({name.replace("Signature", "").lower(): ssgsea_scores(log_tpm, genes) for name, genes in signatures.items()})
    for name, genes in signatures.items():
        print(f"{name}: {len(set(genes) & set(log_tpm.index))} of {len(genes)} genes found in the expression data")
    purity.to_csv(os.path.join(output_dir, "stromal_immune_scores.csv"))
    print(f"Saved stromal_immune_scores.csv: {purity.shape}")
else:
    purity = None
    print(f"SI_geneset.gmt not found in {base_dir}: E4 is NOT adjusted for stromal / immune content")


In [ ]:
# Expression requirement and evidence E4: the TF's own expression

# log2(TPM+1) of the TFs, restricted to patients with survival data
patients = [p for p in log_tpm.columns if p in clinical.index]
tf_expr  = log_tpm.reindex(tf.index).dropna().loc[:, patients]
surv     = clinical.loc[patients, ["OS_MONTHS", "OS_STATUS"]]

# Expression requirement: fraction of tumours where the TF reaches MIN_TPM, and its median TPM
tf["frac_expressed"] = (tf_expr >= np.log2(MIN_TPM + 1)).mean(axis=1).reindex(tf.index)
tf["median_TPM"]     = (2 ** tf_expr.median(axis=1) - 1).reindex(tf.index)
tf["expressed"]      = tf["frac_expressed"] >= MIN_FRAC

# Covariates: z-scored stromal and immune scores, if available
covariates = None
if purity is not None:
    covariates = purity.loc[patients]
    covariates = (covariates - covariates.mean()) / covariates.std()

# E4: Cox per TF on its z-scored expression (HR per 1 sd), only for expressed TFs
# Fitted twice: alone (unadjusted) and with the stromal and immune scores as covariates (adjusted)
# The adjusted result is the one used; without the scores it equals the unadjusted one
rows = {}
for g in tf.index[tf["expressed"].fillna(False)]:
    x = tf_expr.loc[g]
    if x.std() == 0:
        continue
    df_cox = surv.copy()
    df_cox["expr"] = ((x - x.mean()) / x.std()).values
    try:
        cph = CoxPHFitter().fit(df_cox.dropna(), duration_col="OS_MONTHS", event_col="OS_STATUS")
        hr_unadj, p_unadj = cph.hazard_ratios_["expr"], cph.summary.loc["expr", "p"]
        hr_adj, p_adj = hr_unadj, p_unadj
        if covariates is not None:
            cph = CoxPHFitter().fit(df_cox.join(covariates).dropna(), duration_col="OS_MONTHS", event_col="OS_STATUS")
            hr_adj, p_adj = cph.hazard_ratios_["expr"], cph.summary.loc["expr", "p"]
        rows[g] = (hr_adj, p_adj, hr_unadj, p_unadj)
    except Exception:
        pass

own = pd.DataFrame.from_dict(rows, orient="index", columns=["own_HR", "own_p", "own_HR_unadjusted", "own_p_unadjusted"])
for col in own.columns:
    tf[col] = own[col].reindex(tf.index)
tf["own_fdr"]            = bh_fdr(tf["own_p"])
tf["own_fdr_unadjusted"] = bh_fdr(tf["own_p_unadjusted"])
tf["E4_adjusted_for_purity"] = covariates is not None

print(f"Expressed TFs: {int(tf['expressed'].sum())} / {len(tf)} | own expression prognostic (FDR <= 0.05): "
      f"unadjusted {int((tf['own_fdr_unadjusted'] <= 0.05).sum())} | used (adjusted: {covariates is not None}) {int((tf['own_fdr'] <= 0.05).sum())}")


In [ ]:
# Evidence E5: master regulators from 06 (optional; skipped if 06 has not been run)
mr_path  = os.path.join(output_dir, "master_regulators_per_state.csv")
deg_path = os.path.join(output_dir, "tf_network_degrees.csv")

if os.path.exists(mr_path):
    mr = pd.read_csv(mr_path)
    tf["n_states_master"] = mr.groupby("MasterRegulator")["State"].nunique().reindex(tf.index).fillna(0).astype(int)
else:
    tf["n_states_master"] = 0
    print("master_regulators_per_state.csv not found: E5 is 0 for every TF")

if os.path.exists(deg_path):
    tf["out_in_ratio"] = pd.read_csv(deg_path, index_col=0)["out/in"].reindex(tf.index)

print(f"TFs that are a master regulator in at least one state: {int((tf['n_states_master'] > 0).sum())}")

In [ ]:
# Combine the evidence and rank: Robust Rank Aggregation (Kolde et al. 2012)
# The TFs are ranked separately on each evidence line; a TF gets a small p-value if it sits near the top of the lists more
# consistently than expected if the lists were unrelated. No cut-offs are needed and ties are rare

def rank_aggregation(norm_ranks):

    '''

    Robust Rank Aggregation rho score per item.

    Args: norm_ranks (df), items x lists. Rank of each item in each list divided by the number of items (0-1, small = top);
          1.0 where the item has no evidence in that list

    Returns: Series, index = items. p-value (rho score with Bonferroni correction over the number of lists, capped at 1)

    '''

    ordered = np.sort(norm_ranks.values, axis=1)
    n_lists = ordered.shape[1]

    # For the k-th smallest of n_lists uniform ranks, the probability of being <= the observed value is Beta(k, n_lists - k + 1)
    k = np.arange(1, n_lists + 1)
    p_order = beta.cdf(ordered, k, n_lists - k + 1)
    return pd.Series(np.minimum(p_order.min(axis=1) * n_lists, 1.0), index=norm_ranks.index)

# E2 as a continuous quantity: binomial test of risk vs protective among the TF's survival-associated regulons
# (small p = many regulons pointing the same way)
tf["direction_p"] = [binomtest(int(max(r, p)), int(r + p), 0.5).pvalue if (r + p) > 0 else 1.0
                     for r, p in zip(tf["n_risk"], tf["n_protective"])]

# One ranking per evidence line, expressed TFs only: (column, True if a larger value is stronger evidence, value meaning "no evidence")
ranked = tf[tf["expressed"].fillna(False)]
lines = {"E1": ("enrich_p", False, 1.0), "E2": ("direction_p", False, 1.0), "E3": ("n_driver_flows_final", True, 0),
         "E4": ("own_p", False, 1.0), "E5": ("n_states_master", True, 0)}
norm_ranks = pd.DataFrame(index=ranked.index)
for line, (col, larger_is_better, no_evidence) in lines.items():
    values = ranked[col].fillna(no_evidence)
    norm = values.rank(ascending=not larger_is_better, method="average") / len(ranked)
    norm_ranks[line] = norm.where(values != no_evidence, 1.0)
    tf[f"{line}_rank"] = norm_ranks[line].reindex(tf.index)

tf["rra_p"]   = rank_aggregation(norm_ranks).reindex(tf.index)
tf["rra_fdr"] = bh_fdr(tf["rra_p"])

# Descriptive flags (not used for the ranking): which lines pass a conventional threshold
tf["E1_enriched"]         = (tf["enrich_fdr"] <= 0.05) & (tf["n_final"] >= 3)
tf["E2_consistent"]       = (tf["n_survival"] >= 3) & (tf["direction_consistency"] >= 0.8)
tf["E3_driver_linked"]    = tf["n_driver_flows_final"] > 0
tf["E4_own_prognostic"]   = tf["own_fdr"] <= 0.05
tf["E5_master_regulator"] = tf["n_states_master"] > 0

evidence_cols = ["E1_enriched", "E2_consistent", "E3_driver_linked", "E4_own_prognostic", "E5_master_regulator"]
tf["evidence_score"] = tf[evidence_cols].sum(axis=1).astype(int)

# Rank expressed TFs by the aggregation p-value; unexpressed TFs go to the bottom unranked
tf = tf.sort_values(["expressed", "rra_p", "n_final"], ascending=[False, True, False])
tf["rank"] = np.where(tf["expressed"].fillna(False), np.arange(1, len(tf) + 1), np.nan)

print(f"TFs ranked: {len(ranked)} | rank aggregation FDR <= 0.05: {int((tf['rra_fdr'] <= 0.05).sum())} | <= 0.25: {int((tf['rra_fdr'] <= 0.25).sum())}")
print("\nHow often each descriptive flag is met (expressed TFs):")
print(tf[tf["expressed"].fillna(False)][evidence_cols].sum().to_string())

show = ["rra_p", "rra_fdr", "evidence_score", "n_regulons", "n_final", "direction", "upstream_known", "upstream_candidate", "median_TPM", "own_HR", "own_fdr", "n_states_master"]
print("\nTop 25 candidate regulators:")
print(tf.head(25)[show].round(4).to_string())


In [ ]:
# Annotation only: prior evidence for this cancer in Open Targets (does not change the ranking)
# Uses open_targets_associations.csv, saved in the data folder by driver_tiers.py when the driver list was fetched

def classify(score):

    '''

    Labels a gene by how much prior evidence Open Targets has linking it to this cancer.

    Args: score (float). Open Targets association score (0-1); 0 if the gene is not listed for the disease

    Returns: str. "Known (strong)" (>= 0.5), "Known (moderate)" (>= 0.1), "Weak prior evidence" (> 0) or "No prior association" (0)

    '''

    if score >= 0.5: return "Known (strong)"
    if score >= 0.1: return "Known (moderate)"
    if score > 0.0:  return "Weak prior evidence"
    return "No prior association"

assoc_path = os.path.join(data_dir, "open_targets_associations.csv")
if os.path.exists(assoc_path):
    assoc = pd.read_csv(assoc_path).set_index("Gene")
    tf["OT_score"]  = assoc["ot_score"].reindex(tf.index).fillna(0.0)
    tf["OT_status"] = tf["OT_score"].apply(classify)

    # Is the TF itself a published driver of this cancer?
    tf["TF_driver_tier"] = [tier_of.get(g, "none") for g in tf.index]
    print(tf[tf["expressed"].fillna(False)].head(50)["OT_status"].value_counts().to_string())
else:
    print("Open Targets annotation skipped: open_targets_associations.csv not found (driver list came from the fallback, no internet)")

tf.to_csv(os.path.join(output_dir, "candidate_regulators.csv"))
print(f"Saved candidate_regulators.csv: {tf.shape}")


In [ ]:
# Top 20 candidate regulators
# Bar length = number of disease-relevant regulons the TF controls; colour = direction of its survival-associated regulons
# Order = rank aggregation p-value. Text to the right = number of descriptive evidence flags met and the driver mutations upstream of the TF
top = tf[tf["expressed"].fillna(False)].head(20).iloc[::-1]
colours = {"risk": "#D55E00", "protective": "#0072B2", "mixed": "#999999"}

fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(top.index, top["n_final"], color=[colours[d] for d in top["direction"]], edgecolor="grey", linewidth=0.5)
for y, (g, r) in enumerate(top.iterrows()):
    label = f"{int(r['evidence_score'])}/5" + (f"  |  {r['upstream_drivers']}" if r["upstream_drivers"] else "")
    ax.text(r["n_final"] + top["n_final"].max() * 0.01, y, label, va="center", fontsize=9)

ax.set_xlim(0, top["n_final"].max() * 1.35)
ax.set_xlabel("Disease-relevant regulons controlled by the TF")
ax.set_title(f"Top 20 candidate regulators in {CANCER}\n(ranked by Robust Rank Aggregation; label = evidence flags met | upstream driver mutations)", fontweight="bold")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colours.values()],
          labels=["higher activity, worse survival", "higher activity, better survival", "mixed"], loc="lower right", frameon=False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "candidate_regulators_top20.png"), dpi=300, bbox_inches="tight")
plt.show()